# Output Parser
- Output Parser는 LLM의 출력을 처리하여 애플리케이션에서 사용할 수 있는 적절한 형식으로 변환하는 역할을 한다. 
  - LLM이 생성한 결과(Raw text)를 분석하여 **특정 정보를 추출**하거나, **원하는 형식으로 재구성**하는 데 사용된다.
- Output parser를 통해 LLM이 응답하는 **비구조적 데이터를 구조화된 데이터로 변환**하여 후속 작업에 적합하게 만드는 데 사용된다.

## 주요 Output Parser
1. **CommaSeparatedListOutputParser**
    - 쉼표로 구분된 텍스트를 리스트로 변환
2. **JsonOutputParser**
    - JSON 형태로 받은 결과를 JSON 형식으로 변환
3. **PydanticOutputParser**
    - JSON 형태로 받은 결과를 Pydantic 모델 객체로 변환
4. **YamlOutputParser**
    - YAML 형태로 받은 응답을 pydantic 모델객체로 변환.
5. **StrOutputParser**
    - 모델의 출력결과를 문자열로 변환
- PydanticOutputParser, JsonOutputParser, YamlOutputParser는 Pydantic을 이용해 schema를 정의하고 이를 이용해 출력을 변환한다.
  
## 메소드
- parse(text:str)
  - LLM이 생성한 응답(text)을 받아 출력 구조에 맞게 변환하여 반환.
- get_format_instructions(): str
  - output parser가 입력받을 형식으로 LLM이 출력(응답) 하도록 하는 프롬프트를 반환한다.
  - LLM에 전송하는 프롬프트에 포함되어 출력 형식을 안내한다.
- [**Runnable**](05_chaing_LECL.ipynb#Runnable)을 상속받아 구현되어 invoke()를 이용해서 parsing 할 수있다.


In [1]:
from dotenv import load_dotenv
load_dotenv()

True

## StrOutputParser
- 모델(LLM)의 출력 결과를 string으로 변환하여 반환하는 output parser.
- Chat Model은  Message 객체에서 content 속성값을 추출하여 문자열로 반환한다.

In [5]:
from langchain.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
#output parser
from langchain_core.output_parsers import StrOutputParser

prompt_template = ChatPromptTemplate.from_template(
    "한국의 {topic} 관련된 속담을 {count}개 알려줘. 목록 형식으로 작성해줘"
)

model = ChatOpenAI(model='gpt-4o-mini')
#output parser
parse = StrOutputParser()

#prompt 생성
prompt = prompt_template.invoke({'topic':'호랑이', 'count':5})
res = model.invoke(prompt)

print(res)
#parser 이용해서 content만 추출
result = parse.invoke(res)
print(result)

content='물론입니다! 한국의 호랑이와 관련된 속담 5개를 아래에 정리해 드리겠습니다.\n\n1. **호랑이 굴에 가야 호랑이 새끼를 잡는다.**  \n   (위험을 감수해야 큰 성과를 얻을 수 있다는 의미)\n\n2. **호랑이도 제 말 하면 온다.**  \n   (어떤 이야기를 하면 그 대상이 실제로 나타나는 경우를 비유적으로 표현)\n\n3. **호랑이와 나무에 오르기.**  \n   (위험한 상황에서 벗어나기 어렵다는 의미)\n\n4. **호랑이의 가죽은 못 가져가도 제 힘은 뺏지 못한다.**  \n   (실질적인 힘이나 능력은 쉽게 빼앗을 수 없다는 의미)\n\n5. **호랑이가 되어야 소를 잡는다.**  \n   (강한 자가 되어야 목표를 이룰 수 있다는 의미)\n\n이 속담들은 호랑이의 강력한 이미지와 관련된 교훈이나 의미를 담고 있습니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 237, 'prompt_tokens': 30, 'total_tokens': 267, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0705bf87c0', 'finish_reason': 'stop', 'logprobs': None} id='run-72078617-61ca-4596-9d13-2029a5e1ea4a-0' usage_metadata={'input_tokens': 30, 'output_tokens': 2

## CommaSeparatedListOutputParser

- `,`를 구분자로 하는 항목들을 받아서 List로 반환한다.
  - "a,b,c" => ['a','b','c']

In [8]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
parser = CommaSeparatedListOutputParser()
txt = '이순신, 유관순, 강감찬, 안중근'
res = parser.invoke(txt)
print(type(res))


<class 'list'>


In [12]:
from langchain.prompts import PromptTemplate
# output parser가 변환할 수 있는 형식의 문자열을 받기 위한 가이드
# 이것을 프롬포트에 추가한다
print(parser.get_format_instructions())

parser = CommaSeparatedListOutputParser()
prompt_template = PromptTemplate(
    template = '{subject}의 이름 다섯개를 나열해주세요.\n{format_instruction}',
    partial_variables={'format_instruction':parser.get_format_instructions()}  # dictionary 형태 템플릿 변수에 값을 prompt template을 만들면서 넣을 때 사용 (invoke() 사용하지않음)
)
model = ChatOpenAI(model='gpt-4o-mini')

#prompt 생성
#{format_instruction} 변수에는 partial_variables에 설정한 값을 넣음
prompt = prompt_template.invoke({'subject':'축구선수'})
res = model.invoke(prompt)

print(res.content)
result = parser.invoke(res)
print(result)

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
리오넬 메시, 크리스티아누 호날두, 네이마르, 모하메드 살라, 케빈 더 브라위너
['리오넬 메시', '크리스티아누 호날두', '네이마르', '모하메드 살라', '케빈 더 브라위너']


In [13]:
# chain 생성
chain = prompt_template | model | parser

result = chain.invoke({'subject':'야구선수'})
print(result)

['이승엽', '박찬호', '김광현', '류현진', '최형우']


## JsonOutputParser

- JSON 형태로 받은 응답을 dictionary로 반환
- JSON 형식을 정하려는 경우 [Pydantic](Pydantic.ipynb)을 이용해 JSON 스키마를 정의하여 JsonOutputParser 생성시 전달한다.
- LLM이 JSON Schema를 따르는 형태로 응답을 하게 하고 그 것을 JsonOutputParser는 Dictionary로 변환한다.



In [17]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
txt = '''{
    "이름":"홍길동",
    "나이":20,
    "취미":["게임", "독서"]
}'''
res = parser.invoke(txt)
print(res['이름'])

홍길동


In [21]:
parser = JsonOutputParser()

prompt_template = PromptTemplate(
    template='{name}에 대해서 설명해줘.\n{guide}',
    partial_variables={'guide':parser.get_format_instructions()}
)
model = ChatOpenAI(model='gpt-4o-mini')

prompt = prompt_template.invoke({'name':'아이폰'})
print(prompt)
res = model.invoke(prompt)
print(res.content)

print('------------------------------------------')

result = parser.invoke(res)
print(result)

text='아이폰에 대해서 설명해줘.\nReturn a JSON object.'
```json
{
  "product": {
    "name": "아이폰",
    "manufacturer": "애플",
    "first_release_date": "2007-06-29",
    "current_model": "아이폰 15",
    "features": [
      {
        "name": "디스플레이",
        "type": "Retina OLED",
        "size": "6.1 인치"
      },
      {
        "name": "프로세서",
        "type": "A16 Bionic",
        "architecture": "64-bit"
      },
      {
        "name": "카메라",
        "rear": {
          "megapixels": 48,
          "features": ["야간 모드", "4K 비디오 촬영"]
        },
        "front": {
          "megapixels": 12,
          "features": ["슬로모션", "프로필터"]
        }
      },
      {
        "name": "운영 체제",
        "current_version": "iOS 17"
      },
      {
        "name": "배터리 수명",
        "hours": "최대 20시간"
      }
    ],
    "connectivity": [
      "5G",
      "Wi-Fi 6",
      "Bluetooth 5.3"
    ],
    "color_options": ["검정", "흰색", "파란색", "분홍색", "황금색"],
    "storage_options": ["128GB", "256GB", "512GB"]
  }
}
```
-----

In [ ]:
# JSON 문자열의 구조 (schema 정의)
# 알고싶은 정보가 무엇인지 설정
# pydantic 라이브러리 이용해서 정의

In [26]:
# 스키마 정의
from pydantic import BaseModel, Field

class Item(BaseModel) :
    # 스키마를 class 변수로 정의 - 변수명 = key
    name:str = Field(description='제품의 이름')
    info:str = Field(description='제품에 대한 소개 정보')
    price:int = Field(description='제품의 가격')

parser = JsonOutputParser(pydantic_object=Item)
print(parser.get_format_instructions())

prompt_template = PromptTemplate(
    template='{name}에 대해서 설명해줘.\n{guide}',
    partial_variables={'guide':parser.get_format_instructions()}
)
model = ChatOpenAI(model='gpt-4o-mini')

prompt = prompt_template.invoke({'name':'아이폰'})
print(prompt)
res = model.invoke(prompt)

print(res.content)
result = parser.invoke(res)
print(result)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "제품의 이름", "title": "Name", "type": "string"}, "info": {"description": "제품에 대한 소개 정보", "title": "Info", "type": "string"}, "price": {"description": "제품의 가격", "title": "Price", "type": "integer"}}, "required": ["name", "info", "price"]}
```
text='아이폰에 대해서 설명해줘.\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "

## PydanticOutputParser

- JSON 형태로 받은 응답을 Pydantic 모델로 변환하여 반환한다.
- 구현은 JsonOutputParser와 동일한데 parsing 결과를 pydantic 모델로 반환한다.

In [34]:
from langchain_core.output_parsers import PydanticOutputParser

class Person(BaseModel) :
    name:str = Field(description='사람의 이름')
    yob:int = Field(description='name이 태어난 연도')
    yod:int = Field(description='name이 사망한 연도')
    profile:str = Field(description='name에 대한 정보')

parser = PydanticOutputParser(pydantic_object=Person)
print(parser.get_format_instructions())
    
prompt_template = PromptTemplate(
    template='{name}에 대해서 설명해주세요.\n{guide}',
    partial_variables={'guide':parser.get_format_instructions()}
)

prompt = prompt_template.invoke({'name':'베토벤'})
res = model.invoke(prompt)
print(res.content)

result = parser.invoke(res)
print(result)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"name": {"description": "사람의 이름", "title": "Name", "type": "string"}, "yob": {"description": "name이 태어난 연도", "title": "Yob", "type": "integer"}, "yod": {"description": "name이 사망한 연도", "title": "Yod", "type": "integer"}, "profile": {"description": "name에 대한 정보", "title": "Profile", "type": "string"}}, "required": ["name", "yob", "yod", "profile"]}
```
```json
{
  "name": "루트비히 판 베토벤",
  "yob": 1770,
  "yod": 1827,
  "profile": "루트비히 판 베토벤은 독일의 작곡가이자 피아니스트로, 고전주의와 낭만주의 음악의 전환기에 중요한 역할을 하였습니다. 그의 작품은 감정의 깊이와 혁신적인 구조로 유명하며, 교향곡, 피아

## YamlOutputParser
- YAML 형태로 받은 응답을 pydantic Model 객체로 반환한다.

In [40]:
from langchain.output_parsers import YamlOutputParser

# pydantic으로 스키마 정의
class Person(BaseModel) :
    name:str = Field(description='사람의 이름')
    yob:int = Field(description='name이 태어난 연도')
    yod:int = Field(description='name이 사망한 연도')
    profile:str = Field(description='name에 대한 정보')

parser = YamlOutputParser(pydantic_object=Person)
# print(parser.get_format_instructions())
    
prompt_template = PromptTemplate(
    template='{name}에 대해서 설명해주세요.\n{guide}',
    partial_variables={'guide':parser.get_format_instructions()}
)

prompt = prompt_template.invoke({'name':'모차르트'})
res = model.invoke(prompt)
print(res.content)

result = parser.invoke(res)
print(result)

```
name: Wolfgang Amadeus Mozart
yob: 1756
yod: 1791
profile: A prolific and influential composer of the Classical era, known for his symphonies, operas, and chamber music.
```
name='Wolfgang Amadeus Mozart' yob=1756 yod=1791 profile='A prolific and influential composer of the Classical era, known for his symphonies, operas, and chamber music.'
